In [1]:
from brollm import BaseContract
from broflow import BaseTask, TaskRegistry, Flow
from broskill import SkillControl, ToolControl, Skill, Tool, Arg
from broskill.processing.tool import to_args

from pathlib import Path
import subprocess
import sys
from enum import StrEnum
from typing import Any
from dataclasses import dataclass, field

In [2]:
class Process(StrEnum):
    INPUT = "input"
    ROUTER = "router"
    TOOL_USE = "tool_use"
    FOLLOW_UP_QUESTION = "ask_follow_up_question"
    ANSWER = "answer"
    FAILED_RECOVERY = "failed_recovery"
    END = "end"


MAX_RETRIES = 3


@dataclass
class State:
    messages: list = field(default_factory=list)
    input: str = ''
    next_state: Process = Process.INPUT
    follow_up_questions: str = ''
    tool_calls: list[dict[str, Any]] = field(default_factory=list)
    tool_results: list[dict[str, Any]] = field(default_factory=list)
    error_message: str = ''
    retry_count: int = 0
    skills: list = field(default_factory=list)
    answer: str = ''


class UserInput(BaseTask):
    possible_next = {Process.ROUTER}

    def __call__(self, state: State):
        state.input = input(state.follow_up_questions)
        state.follow_up_questions = ''
        self.set_next(Process.ROUTER)
        return state


class Router(BaseTask):
    """Stand-in for the real structured-output LLM call: reads state.next_state
    as a manually-set decision so the flow can be studied/traced without a
    live model. Swap the body for a real llm() call later -- the contract
    (read state, call set_next) stays the same."""
    possible_next = {Process.ANSWER, Process.TOOL_USE, Process.FOLLOW_UP_QUESTION, Process.FAILED_RECOVERY}

    def __call__(self, state: State):
        try:
            has_succeeded = any(r.get("success") for r in state.tool_results)
            if state.next_state == Process.TOOL_USE and not has_succeeded:
                self.set_next(Process.TOOL_USE)
                if not state.tool_calls:
                    state.tool_calls.append({"name": "search", "input": {"query": state.input}})
            elif state.next_state == Process.FOLLOW_UP_QUESTION:
                self.set_next(Process.FOLLOW_UP_QUESTION)
            else:
                self.set_next(Process.ANSWER)
        except Exception as e:
            state.error_message = f"router failed: {e}"
            self.set_next(Process.FAILED_RECOVERY)
        return state


class ToolUse(BaseTask):
    """Tools return only text. Any call whose input carries force_fail=True
    simulates a failure, purely so FailedRecovery's path can be exercised on
    purpose while studying -- delete that flag once real tools are wired in."""
    possible_next = {Process.ROUTER, Process.FAILED_RECOVERY}

    def __call__(self, state: State):
        any_failed = False
        for call in state.tool_calls:
            failed = bool(call.get("input", {}).get("force_fail"))
            state.tool_results.append({
                "name": call["name"],
                "input": call["input"],
                "output": f"tool '{call['name']}' failed: simulated failure" if failed else "mock result",
                "success": not failed,
            })
            any_failed = any_failed or failed
        state.tool_calls = []

        if any_failed and state.retry_count < MAX_RETRIES:
            state.error_message = next(r["output"] for r in state.tool_results if not r["success"])
            self.set_next(Process.FAILED_RECOVERY)
        else:
            self.set_next(Process.ROUTER)
        return state


class FailedRecovery(BaseTask):
    """Given the last failure, decide whether to retry with corrected args or
    give up and let Answer explain. Deliberately narrower than Router: it only
    ever reasons about the one call that just failed, not which capability to
    use next -- that's why it goes straight back to TOOL_USE, not ROUTER."""
    possible_next = {Process.TOOL_USE, Process.ANSWER}

    def __call__(self, state: State):
        state.retry_count += 1
        if state.retry_count >= MAX_RETRIES:
            state.answer = f"I couldn't complete this after {state.retry_count} attempts: {state.error_message}"
            self.set_next(Process.ANSWER)
            return state

        last_failed = next(r for r in reversed(state.tool_results) if not r["success"])
        corrected_input = {k: v for k, v in last_failed["input"].items() if k != "force_fail"}
        state.tool_calls.append({"name": last_failed["name"], "input": corrected_input})
        self.set_next(Process.TOOL_USE)
        return state


class FollowUpQuestion(BaseTask):
    """This will have a special prompt designed to clarify and return as text"""
    possible_next = {Process.INPUT}

    def __call__(self, state: State):
        state.follow_up_questions = 'Based on the information you provided, could you clarify or expand on anything?'
        self.set_next(Process.INPUT)
        return state


class Answer(BaseTask):
    """If all things done, return answer as text"""
    possible_next = {Process.END}

    def __call__(self, state: State):
        if not state.answer:
            state.answer = "Here is your answer. You're AWESOME!"
        self.set_next(Process.END)
        return state

In [3]:
registry = TaskRegistry()
registry.register(Process.INPUT, UserInput(Process.INPUT.value))
registry.register(Process.ROUTER, Router(Process.ROUTER.value))
registry.register(Process.TOOL_USE, ToolUse(Process.TOOL_USE.value))
registry.register(Process.FAILED_RECOVERY, FailedRecovery(Process.FAILED_RECOVERY.value))
registry.register(Process.FOLLOW_UP_QUESTION, FollowUpQuestion(Process.FOLLOW_UP_QUESTION.value))
registry.register(Process.ANSWER, Answer(Process.ANSWER.value))

flow = Flow(registry)

## Three traceable demos, no real LLM/input() needed

Each constructs a `State` by hand (standing in for what a real `Router` LLM
call would have decided) and runs the flow from `ROUTER` straight through to
`END`, so the mechanics are visible in `flow.trace` without needing to type
anything into `input()`.

In [4]:
# Demo A -- happy path: one tool call, no failure
state = State(input="what's the weather in Paris?", next_state=Process.TOOL_USE)
flow.run(start=Process.ROUTER, end=Process.END, state=state)

print("trace:", flow.trace)
print("answer:", state.answer)

trace: [('router', <Process.TOOL_USE: 'tool_use'>), ('tool_use', <Process.ROUTER: 'router'>), ('router', <Process.ANSWER: 'answer'>), ('answer', <Process.END: 'end'>)]
answer: Here is your answer. You're AWESOME!


In [5]:
# Demo B -- the tool fails once (force_fail=True), FailedRecovery strips the
# flag and retries the SAME tool, which then succeeds
state = State(
    input="what's the weather in Paris?",
    next_state=Process.TOOL_USE,
    tool_calls=[{"name": "search", "input": {"query": "paris weather", "force_fail": True}}],
)
flow.run(start=Process.ROUTER, end=Process.END, state=state)

print("trace:", flow.trace)
print("retry_count:", state.retry_count)
print("tool_results:", state.tool_results)
print("answer:", state.answer)

trace: [('router', <Process.TOOL_USE: 'tool_use'>), ('tool_use', <Process.FAILED_RECOVERY: 'failed_recovery'>), ('failed_recovery', <Process.TOOL_USE: 'tool_use'>), ('tool_use', <Process.ROUTER: 'router'>), ('router', <Process.ANSWER: 'answer'>), ('answer', <Process.END: 'end'>)]
retry_count: 1
tool_results: [{'name': 'search', 'input': {'query': 'paris weather', 'force_fail': True}, 'output': "tool 'search' failed: simulated failure", 'success': False}, {'name': 'search', 'input': {'query': 'paris weather'}, 'output': 'mock result', 'success': True}]
answer: Here is your answer. You're AWESOME!


In [6]:
# Demo C -- retries exhausted: seed retry_count one below MAX_RETRIES so a
# single failure pushes it over the cap, falling through to an honest answer
state = State(
    input="what's the weather in Paris?",
    next_state=Process.TOOL_USE,
    tool_calls=[{"name": "search", "input": {"query": "paris weather", "force_fail": True}}],
    retry_count=MAX_RETRIES - 1,
)
flow.run(start=Process.ROUTER, end=Process.END, state=state)

print("trace:", flow.trace)
print("retry_count:", state.retry_count)
print("answer:", state.answer)

trace: [('router', <Process.TOOL_USE: 'tool_use'>), ('tool_use', <Process.FAILED_RECOVERY: 'failed_recovery'>), ('failed_recovery', <Process.ANSWER: 'answer'>), ('answer', <Process.END: 'end'>)]
retry_count: 3
answer: I couldn't complete this after 3 attempts: tool 'search' failed: simulated failure


## Real interactive run

Starts from `INPUT`, so it actually pauses on `input()` -- unlike the three
demos above, this one needs you to type something. `Router`'s decision is
still the mock (driven by `next_state`, which you'd set to `TOOL_USE` before
running to see it reach that branch) -- swap that body for a real LLM call
once you're ready to stop hand-driving it.

In [7]:
state = State(next_state=Process.ANSWER)  # set to Process.TOOL_USE to exercise that branch interactively
flow.run(start=Process.INPUT, end=Process.END, state=state)

print("trace:", flow.trace)
print("answer:", state.answer)

trace: [('input', <Process.ROUTER: 'router'>), ('router', <Process.ANSWER: 'answer'>), ('answer', <Process.END: 'end'>)]
answer: Here is your answer. You're AWESOME!


# Test model idea

In [8]:
MODEL_LIST = [
    "google.gemma-3-4b-it",
    "google.gemma-3-12b-it", # support tool use
    "google.gemma-3-27b-it"
]

In [9]:
import boto3

model = boto3.client('bedrock-runtime', region_name='us-east-1')

In [31]:
ROOT = Path().cwd().resolve().parent
SKILL_DIR = ROOT / "skills"
sc = SkillControl(SKILL_DIR)
tc = ToolControl(sc)

In [32]:
sc.list_skills()
print(sc.load_skill('tool-call'))

# Tool Call

Native tool-calling is a model feature, not a guarantee -- some models don't support
it, and support can change without notice. This skill replaces that feature with a
plain-text contract: tools are described in the prompt, and the model's entire
response is parsed as JSON. Reliability comes from the contract being narrow and
strictly enforced by the parsing code, not from trusting the model's judgment.

This skill's body is static -- it never changes between calls. The caller appends
an `## Available Skills` section and an `## Available Tools` section after it,
built fresh each call from whatever is actually registered at the time. Those two
section headers are the contract between this file and the caller: don't rename
one without renaming the other.

## Instructions

- Read the Available Skills and Available Tools sections appended below this file.
  Available Tools entries each give a tool's name, description, and the input
  schema it requires.
- Decide which skill or

In [ ]:
available_skills = [f"- {s.name}: {s.description}" for s in sc.list_skills() if s.name != 'tool-call']

["- ask-followup-question: Ask the user a clarifying question when you don't have enough information to proceed safely or correctly. Use only when something is genuinely missing or ambiguous -- never speculatively, and never for something you could reasonably infer from context already given.",
 "- read-file: Read the contents of a specific file, or list files matching a pattern. Use when the user wants to see what's in a file, or wants to find files matching a pattern.",
 '- tell-joke: Tell a joke on request -- dad jokes, puns, or a mix of both. Use when the user asks for a joke, wants to be entertained, or needs a laugh.']

In [ ]:
MODEL_ID = 'google.gemma-3-4b-it'
# MODEL_ID = 'google.gemma-3-12b-it'
# MODEL_ID = 'google.gemma-3-27b-it'

available_skills = [f"- {s.name}: {s.description}" for s in sc.list_skills() if s.name != 'tool-call']
tool_call_instruction = sc.load_skill('tool-call')
system_prompt = f"""\

# System Prompt
You are an expert AI assistant who is good at identifying the correct skill to use for a given task.

{tool_call_instruction}

## Available Skills:
{"\n".join(available_skills)}

## Available Tools:

""".strip()
messages = [
    {
        "role": "user",
        "content": [{"text": "Tell me a joke!"}]
    }
]
kwargs = {
    "modelId": MODEL_ID,
    "system": [{"text": system_prompt}],
    "messages": messages,
}
response = model.converse(**kwargs)

In [25]:
print(system_prompt)

# System Prompt
You are an expert AI assistant who is good at identifying the correct skill to use for a given task.

# Tool Call

## Instructions

- Select available tools that match users' request from Available Tools section

## Available Tools

```yaml
type: function
function:
  name: load_skill
  description: >-
    Load the full instructions for a registered skill by name.
    Call this only when the current task clearly matches that
    skill's description.
  parameters:
    type: object
    properties:
      skill_name:
        type: string
    required:
      - skill_name
```

## Response

Always response in JSON codeblock start with `json and end with `.

```json
<your response goes here>
```

## Available Skills:
- ask-followup-question: Ask the user a clarifying question when you don't have enough information to proceed safely or correctly. Use only when something is genuinely missing or ambiguous -- never speculatively, and never for something you could reasonably infer fr

In [24]:
response['usage']

{'inputTokens': 481, 'outputTokens': 32, 'totalTokens': 513}

In [22]:
response['output']['message']

{'role': 'assistant',
 'content': [{'text': '```json\n{\n  "tool_call": {\n    "tool_name": "tell-joke"\n  }\n}\n```'}]}

In [ ]:
MODEL_ID = 'google.gemma-3-4b-it'
# MODEL_ID = 'google.gemma-3-12b-it'
# MODEL_ID = 'google.gemma-3-27b-it'
system_prompt = "you're a helpful assistant"
messages = [
    {
        "role": "user",
        "content": [{"text": "Google 'AI Agent' for me."}]
    }
]
kwargs = {
    "modelId": MODEL_ID,
    "system": [{"text": system_prompt}],
    "messages": messages,
}
tool_registry = [
    {
        "toolSpec": {
            "name": "search",
            "description": "use this tool to search the web for information",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "the query to search for"
                        }
                    },
                    "required": ["query"]
                }
            }
        }
    }
]
kwargs["toolConfig"] = {"tools": tool_registry}
response = model.converse(**kwargs)

In [13]:
response

{'ResponseMetadata': {'RequestId': '38bef61a-afd1-4e4e-915e-04e81e8a7b0c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 14 Sep 2026 13:43:42 GMT',
   'content-type': 'application/json',
   'content-length': '5869',
   'connection': 'keep-alive',
   'x-amzn-requestid': '38bef61a-afd1-4e4e-915e-04e81e8a7b0c'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': 'Okay, here\'s a summary of what Google shows for "AI Agent" as of today, November 2, 2023. It\'s a rapidly evolving field, so this is a snapshot in time!  I\'ll break it down into what they *are*, what they *do*, key players, and where things are *going*.\n\n**What *is* an AI Agent?**\n\nAt its core, an AI Agent is a type of artificial intelligence that can **perceive its environment and take actions to achieve a specific goal.**  Think of it as a more sophisticated chatbot or virtual assistant.  However, they go beyond just responding to prompts; they can *autonomously* plan an

In [ ]:
load_skill_tool = Tool(
    name="load_skill",
    description="Load the full instructions for a registered skill by name. Call this only when the current task clearly matches that skill's description.",
    args=[
        Arg(name="skill_name", type="string", required=True)
    ],
    path=Path()
)

load_skill_extension_tool = Tool(
    name="load_skill_extension",
    description="Load the full contents of one reference/asset document belonging to an already-loaded skill. Call this only when that skill's instructions point you to a specific reference/asset file for more detail — don't call it speculatively.",
    args=[
        Arg(name="skill_name", type="string", required=True, description="The name of the skill whose reference/asset you want to load."),
        Arg(name="path", type="string", required=True, description="The reference/asset file's path exactly as shown in the skill's instructions, e.g. `references/aws.md` or `assets/color.ts`.")
    ],
    path=Path()
)

ask_followup_question_tool = Tool(
    name="ask_followup_question",
    description="Signal that you need the user to answer something before you can continue. Write the actual question as your normal response content, then call this tool with no arguments to pause the turn and wait for their reply.",
    args=[],
    path=Path()
)

In [ ]:
def pack_arg(arg:Arg):
    _base = {"type": arg.type}
    if arg.description:
        _base["description"] = arg.description
    return _base

def tool_to_yaml(tool:Tool):
    fn_metadata = dict(
        name=tool.name,
        description=tool.description,
        parameters={
            "type": "object",
            "properties": {arg.name: pack_arg(arg) for arg in tool.args},
            "required": [arg.name for arg in tool.args if arg.required==True]
        }
    )
    return {"type": "function", "function": fn_metadata}

In [55]:
import yaml

tools = [
   tool_to_yaml(t)
   for t in [
      load_skill_tool,
      load_skill_extension_tool,
      ask_followup_question_tool
   ]
]

available_tools = yaml.dump(tools, sort_keys=False)
print(available_tools)

- type: function
  function:
    name: load_skill
    description: Load the full instructions for a registered skill by name. Call this
      only when the current task clearly matches that skill's description.
    parameters:
      type: object
      properties:
        skill_name:
          type: string
      required:
      - skill_name
- type: function
  function:
    name: load_skill_extension
    description: "Load the full contents of one reference/asset document belonging\
      \ to an already-loaded skill. Call this only when that skill's instructions\
      \ point you to a specific reference/asset file for more detail \u2014 don't\
      \ call it speculatively."
    parameters:
      type: object
      properties:
        skill_name:
          type: string
          description: The name of the skill whose reference/asset you want to load.
        path:
          type: string
          description: The reference/asset file's path exactly as shown in the skill's
           

In [69]:
MODEL_ID = 'google.gemma-3-4b-it'
# MODEL_ID = 'google.gemma-3-12b-it'
# MODEL_ID = 'google.gemma-3-27b-it'

available_skills = [f"- {s.name}: {s.description}" for s in sc.list_skills() if s.name not in ['skill-call', 'tool-call']]
skill_call_instruction = sc.load_skill('skill-call')
# tool_call_instruction = sc.load_skill('tool-call')
system_prompt = f"""\
{skill_call_instruction}

## Available Skills
{"\n".join(available_skills)}

## Available Tools
{available_tools}
""".strip()
messages = [
    {
        "role": "user",
        "content": [{"text": "I wanna know what files are in folder skills, and what is in each file?"}]
    }
]
kwargs = {
    "modelId": MODEL_ID,
    "system": [{"text": system_prompt}],
    "messages": messages,
}
response = model.converse(**kwargs)

In [70]:
response

{'ResponseMetadata': {'RequestId': '4a7a14d1-0d39-4fff-a843-efe902fe934c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 16 Sep 2026 15:51:11 GMT',
   'content-type': 'application/json',
   'content-length': '252',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4a7a14d1-0d39-4fff-a843-efe902fe934c'},
  'RetryAttempts': 1},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': '```json\n{"skill_names": ["read-file"]}\n```'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 708, 'outputTokens': 16, 'totalTokens': 724},
 'metrics': {'latencyMs': 304}}

In [71]:
print(system_prompt)

# Skill Call

Pick skills only -- never a tool. This runs before `tool-call`; only skill names
and one-line descriptions are visible here, not tools. The caller appends an
`## Available Skills` section below, built fresh each call from registered skills.

## Instructions

- Pick every skill that genuinely matches the request -- none, one, or several.
- Never invent a skill name not listed in Available Skills.

## Response

Exactly one JSON codeblock, nothing else -- no prose, no reasoning. One key,
`skill_names`, a list of strings. Always present, `[]` when nothing matches.

```json
{"skill_names": ["read-file"]}
```

```json
{"skill_names": []}
```

## Available Skills
- ask-followup-question: Ask the user a clarifying question when you don't have enough information to proceed safely or correctly. Use only when something is genuinely missing or ambiguous -- never speculatively, and never for something you could reasonably infer from context already given.
- read-file: Read the content

In [72]:
MODEL_ID = 'google.gemma-3-4b-it'
# MODEL_ID = 'google.gemma-3-12b-it'
# MODEL_ID = 'google.gemma-3-27b-it'

# available_skills = [f"- {s.name}: {s.description}" for s in sc.list_skills() if s.name not in ['skill-call', 'tool-call']]
# skill_call_instruction = sc.load_skill('skill-call')

tools = [
   tool_to_yaml(t)
   for t in [
        tc.load_tool('read-file', 'scripts/list_files.py'),
        tc.load_tool('read-file', 'scripts/read_file.py')
   ]
]

available_tools = yaml.dump(tools, sort_keys=False)
tool_call_instruction = sc.load_skill('tool-call')
system_prompt = f"""\
{tool_call_instruction}


## Available Tools
{available_tools}
""".strip()

messages = [
    {
        "role": "user",
        "content": [{"text": "I wanna know what files are in folder skills, and what is in each file?"}]
    }
]
kwargs = {
    "modelId": MODEL_ID,
    "system": [{"text": system_prompt}],
    "messages": messages,
}
response = model.converse(**kwargs)

In [73]:
response

{'ResponseMetadata': {'RequestId': '4fe31dca-994c-4872-ae54-1221b31db69c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 16 Sep 2026 15:56:19 GMT',
   'content-type': 'application/json',
   'content-length': '380',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4fe31dca-994c-4872-ae54-1221b31db69c'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': '```json\n{"tool_use": [{"name": "list_files", "input": {"pattern": "skills/*"}}, {"name": "read_file", "input": {"pattern": "skills/list_files/*"}}]}\n```'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 451, 'outputTokens': 53, 'totalTokens': 504},
 'metrics': {'latencyMs': 879}}

In [74]:
print(system_prompt)

# Tool Call

Pick tools only -- never a skill name. Skill selection already happened in an
earlier `skill-call` step; the caller appends an `## Available Tools` section
below, built fresh each call from the currently-loaded skill's own tools.

## Instructions

- Pick every tool that matches the request -- none, one, or several.
- Never invent a tool name or input field not listed in Available Tools.
- Missing or ambiguous input: use `ask_followup_question` instead of guessing.

## Response

Exactly one JSON codeblock, nothing else -- no prose, no reasoning. One key,
`tool_use`, a list of `{"name": ..., "input": {...}}`. Always present, `[]` when
nothing matches.

```json
{"tool_use": [{"name": "read_file", "input": {"pattern": "skills/read-file/SKILL.md"}}]}
```

```json
{"tool_use": []}
```


## Available Tools
- type: function
  function:
    name: list_files
    description: List every file matching a glob pattern, relative to the project
      root.
    parameters:
      type: obje

In [81]:
@dataclass
class TestIdea:
    a:list

    @property
    def a_str(self):
        return "".join(self.a)

In [82]:
TestIdea(a=["1","2"]).a_str

'12'